*0.3 Classical NLP*

# n-grams

**The situation.** A classifier flags urgent tickets from single-word features. "not working" and "working" both contain "working"; "no refund" and "refund" both contain "refund". The model cannot see the "not", because it sees words one at a time and in any order.

**n-grams.** Sequences of *n* neighbouring words treated as one feature: bigrams ("not working", "no refund"), trigrams ("card charged twice"). They put a little word order back into a bag-of-words model. Cheap, and often the biggest single accuracy gain in a TF-IDF classifier.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tickets = ["the app is not working", "the app is working", "no refund received", "refund received"]

for n in (1, 2):
    vectorizer = TfidfVectorizer(ngram_range=(1, n))
    vectorizer.fit(tickets)
    features = vectorizer.get_feature_names_out()
    print(f"ngram_range=(1, {n}): {len(features)} features")
    negations = []
    for feature in features:
        if feature.startswith("not") or feature.startswith("no"):
            negations.append(feature)
    print("  ", negations)
bigram_features = TfidfVectorizer(ngram_range=(1, 2)).fit(tickets).get_feature_names_out()
assert "not working" in bigram_features and "no refund" in bigram_features

ngram_range=(1, 1): 8 features
   ['no', 'not']
ngram_range=(1, 2): 15 features
   ['no', 'no refund', 'not', 'not working']


**Reading the output.** With single words, "not" and "no" are features on their own, unattached. With bigrams, "not working" and "no refund" become features, so a model can learn that *those* mean trouble while "working" and "refund" alone do not.

**Do they help? Measure.** A quick classifier on the 20 Newsgroups posts, unigrams vs unigrams + bigrams.

In [3]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

topics = ["sci.med", "sci.space", "rec.autos"]
train = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
test = fetch_20newsgroups(
    subset="test", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)

results = {}
for n in (1, 2):
    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, n), sublinear_tf=True, min_df=2),
        LogisticRegression(C=10, max_iter=3000),
    )
    model.fit(train.data, train.target)
    features = len(model.named_steps["tfidfvectorizer"].get_feature_names_out())
    results[n] = model.score(test.data, test.target)
    print(f"ngram_range=(1, {n}): {features:>7} features, test accuracy {results[n]:.1%}")

ngram_range=(1, 1):   11525 features, test accuracy 87.4%


ngram_range=(1, 2):   42123 features, test accuracy 86.3%


**Reading the output.** Bigrams multiply the feature count, and accuracy can move either way — up a point or two on some data, down on others (here the extra rare features slightly hurt). It is a setting to measure, not to assume.

**The rule to remember.** `ngram_range=(1, 2)` is the first thing to try after a unigram baseline. Beyond trigrams the features become too rare to help.

| Use it when | Don't when | Instead use |
|---|---|---|
| TF-IDF/BM25 classifiers and search where phrases matter | the corpus is tiny (bigrams are seen once and never again) | unigrams; `min_df=2` to drop one-off n-grams |

**Watch out**
- Feature count explodes; set `min_df` and consider `max_features` to keep the model small.
- Character n-grams (`analyzer="char_wb"`) handle typos and product codes better than word n-grams.
- Neural models see order natively; n-grams are a classical-pipeline tool.